In [1]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.tools import StructuredTool
from langchain_core.messages import HumanMessage, ToolMessage

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)

print("Groq model initialized successfully.")

Groq model initialized successfully.


In [2]:
class ExpenseSplitInput(BaseModel):
    total_amount: float = Field(
        gt=0,
        description="Total bill amount in rupees, greater than zero"
    )
    num_people: int = Field(
        gt=0,
        description="Number of people sharing the bill, a positive integer"
    )


print("Expense splitter input schema created.")
print(ExpenseSplitInput.model_json_schema())

Expense splitter input schema created.
{'properties': {'total_amount': {'description': 'Total bill amount in rupees, greater than zero', 'exclusiveMinimum': 0, 'title': 'Total Amount', 'type': 'number'}, 'num_people': {'description': 'Number of people sharing the bill, a positive integer', 'exclusiveMinimum': 0, 'title': 'Num People', 'type': 'integer'}}, 'required': ['total_amount', 'num_people'], 'title': 'ExpenseSplitInput', 'type': 'object'}


In [3]:
def split_expense(total_amount: float, num_people: int) -> str:
    """Split a total bill equally among a given number of people."""

    expense = ExpenseSplitInput(
        total_amount=total_amount,
        num_people=num_people
    )

    amount_per_person = expense.total_amount / expense.num_people

    return (
        f"Total bill: ₹{expense.total_amount:.2f}\n"
        f"Number of people: {expense.num_people}\n"
        f"Amount per person: ₹{amount_per_person:.2f}"
    )


print("Expense splitting function created.")

Expense splitting function created.


In [4]:
expense_tool = StructuredTool.from_function(
    func=split_expense,
    name="split_expense",
    description=(
        "Split a total bill equally among a given number of people. "
        "Provide total_amount in rupees and num_people."
    ),
    args_schema=ExpenseSplitInput
)

print("StructuredTool created successfully.")
print("Tool name:", expense_tool.name)

StructuredTool created successfully.
Tool name: split_expense


In [5]:
manual_inputs = {
    "total_amount": 2400,
    "num_people": 4
}

manual_result = expense_tool.invoke(manual_inputs)

print("Manual tool result:")
print(manual_result)

Manual tool result:
Total bill: ₹2400.00
Number of people: 4
Amount per person: ₹600.00


In [6]:
llm_with_tools = llm.bind_tools([expense_tool])

user_query = "Split a bill of ₹2400 equally between 4 friends. How much does each person owe?"

messages = [
    HumanMessage(content=user_query)
]

response = llm_with_tools.invoke(messages)

print("User query:")
print(user_query)

print("\nModel-suggested tool calls:")

for call in response.tool_calls:
    print("Tool:", call["name"])
    print("Arguments:", call["args"])

if not response.tool_calls:
    raise RuntimeError(
        "The model did not suggest a tool call. Run this cell again."
    )

User query:
Split a bill of ₹2400 equally between 4 friends. How much does each person owe?

Model-suggested tool calls:
Tool: split_expense
Arguments: {'num_people': 4, 'total_amount': 2400}


In [7]:
messages.append(response)

tool_results = []

for call in response.tool_calls:
    if call["name"] != "split_expense":
        raise ValueError(f"Unexpected tool: {call['name']}")

    result = expense_tool.invoke(call["args"])
    tool_results.append(result)

    print("Executed tool:", call["name"])
    print("Arguments:", call["args"])
    print("\nBound-flow tool result:")
    print(result)

    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"]
        )
    )

print("\nBound-flow tool execution completed.")

Executed tool: split_expense
Arguments: {'num_people': 4, 'total_amount': 2400}

Bound-flow tool result:
Total bill: ₹2400.00
Number of people: 4
Amount per person: ₹600.00

Bound-flow tool execution completed.


In [8]:
final_response = llm.invoke(messages)

print("Final answer from the model:")
print(final_response.content)

Final answer from the model:
Each friend should pay **₹600**.


In [9]:
import re

def extract_amount_per_person(result: str) -> float:
    match = re.search(
        r"Amount per person:\s*₹([\d,]+(?:\.\d{1,2})?)",
        result
    )

    if not match:
        raise ValueError(
            "Could not find the amount per person in the tool result."
        )

    return float(match.group(1).replace(",", ""))


manual_amount = extract_amount_per_person(manual_result)

assert tool_results, "No tool results were generated."

bound_amounts = [
    extract_amount_per_person(result)
    for result in tool_results
]

print(f"Manual result per person: ₹{manual_amount:.2f}")
print("Bound-flow results per person:")

for amount in bound_amounts:
    print(f"₹{amount:.2f}")

assert all(
    abs(amount - manual_amount) < 0.001
    for amount in bound_amounts
), "Manual and bound-flow results do not match."

assert abs(manual_amount - 600.00) < 0.001

print("\nBoth approaches returned the same amount.")

Manual result per person: ₹600.00
Bound-flow results per person:
₹600.00

Both approaches returned the same amount.


In [10]:
assert response.tool_calls, "No tool call was suggested."

assert any(
    call["name"] == "split_expense"
    for call in response.tool_calls
), "The expense splitter tool was not called."

assert final_response.content.strip(), "Final answer is empty."

assert abs(manual_amount - 600.00) < 0.001

assert all(
    abs(amount - 600.00) < 0.001
    for amount in bound_amounts
)

print("Validation passed.")
print("StructuredTool with Pydantic args_schema works.")
print("Manual tool execution succeeded.")
print("The model suggested the expense splitter tool.")
print("Manual and bound-flow calculations match.")
print("The final answer was generated.")

Validation passed.
StructuredTool with Pydantic args_schema works.
Manual tool execution succeeded.
The model suggested the expense splitter tool.
Manual and bound-flow calculations match.
The final answer was generated.


In [11]:
from pydantic import ValidationError

invalid_inputs = [
    {"total_amount": 0, "num_people": 4},
    {"total_amount": 2400, "num_people": 0},
    {"total_amount": -100, "num_people": 2}
]

for index, invalid_input in enumerate(invalid_inputs, start=1):
    try:
        ExpenseSplitInput(**invalid_input)
        raise AssertionError(
            f"Invalid input {index} was incorrectly accepted."
        )
    except ValidationError:
        print(f"Invalid input {index} correctly rejected.")

print("All invalid-input tests passed.")

Invalid input 1 correctly rejected.
Invalid input 2 correctly rejected.
Invalid input 3 correctly rejected.
All invalid-input tests passed.


In [12]:
print("Lab 6 - Exercise 6 completed successfully.")
print("Tool method: StructuredTool with Pydantic args_schema.")
print("Manual result: ₹600.00 per person.")
print("Bound-flow result: ₹600.00 per person.")
print("Both calculation results match.")

Lab 6 - Exercise 6 completed successfully.
Tool method: StructuredTool with Pydantic args_schema.
Manual result: ₹600.00 per person.
Bound-flow result: ₹600.00 per person.
Both calculation results match.
